In [1]:
#1.	Create Pyspark Dataframe using multiple lists.
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("DF Example").getOrCreate()

# Your data
list_A = [1, 2, 3, 4, 5]
list_B = [4, 5, 6, 7, 8]

# Create DataFrame with column names
df = spark.createDataFrame(list(zip(list_A, list_B)), schema=["A", "B"])
df.show()


+---+---+
|  A|  B|
+---+---+
|  1|  4|
|  2|  5|
|  3|  6|
|  4|  7|
|  5|  8|
+---+---+



In [2]:
# Output: [1,2,3]
list_A = [1, 2, 3, 4, 5]
list_B = [4, 5, 6, 7, 8]

result = [item for item in list_A if item not in list_B]
display(result)

[1, 2, 3]

In [4]:
#4. for the below Dataframe , find the in a column find the frequency counts of unique items
data = [("John", "Engineer"), ("John", "Engineer"), ("Mary", "Scientist"),
        ("Bob", "Engineer"), ("Bob", "Engineer"), ("Bob", "Scientist"), ("Sam", "Doctor")]
columns = ['Name', 'Job']
df1 = spark.createDataFrame(data,columns)
df1.show()

+----+---------+
|Name|      Job|
+----+---------+
|John| Engineer|
|John| Engineer|
|Mary|Scientist|
| Bob| Engineer|
| Bob| Engineer|
| Bob|Scientist|
| Sam|   Doctor|
+----+---------+



In [6]:
df1.groupby('Job').count().show()

+---------+-----+
|      Job|count|
+---------+-----+
| Engineer|    4|
|Scientist|    2|
|   Doctor|    1|
+---------+-----+



In [10]:
#Output : keep only top 2 most frequent values as it is and replace everything else as ‘Other’
from pyspark.sql import functions as F

top2_jobs = [row['job'] for row in df1.groupBy("job").count().orderBy(F.desc("count")).limit(2).collect()]

df_top2 = df1.withColumn(
    "job",
    F.when(F.col("job").isin(top2_jobs), F.col("job")).otherwise(F.lit("Other"))
)
df_top2.show()


+----+---------+
|Name|      job|
+----+---------+
|John| Engineer|
|John| Engineer|
|Mary|Scientist|
| Bob| Engineer|
| Bob| Engineer|
| Bob|Scientist|
| Sam|    Other|
+----+---------+



In [13]:
#6. rename columns of a PySpark DataFrame using two lists – one containing the old column
#   names and the other containing the new column names

old_names = ["col1", "col2", "col3"]

# new column names
new_names = ["new_col1", "new_col2", "new_col3"]

data = [(1, 2, 3), (4, 5, 6)]
df2 = spark.createDataFrame(data, old_names)

df2.show()

cols_map = dict(zip(old_names,new_names))
df_renamed = df2.withColumnsRenamed(cols_map)
df_renamed.show()


+----+----+----+
|col1|col2|col3|
+----+----+----+
|   1|   2|   3|
|   4|   5|   6|
+----+----+----+

+--------+--------+--------+
|new_col1|new_col2|new_col3|
+--------+--------+--------+
|       1|       2|       3|
|       4|       5|       6|
+--------+--------+--------+



In [19]:
#7. find the numbers that are multiples of 3 from a column
from pyspark.sql.functions import col, when
data = [(0, 7), (1, 6), (2, 9), (3, 7), (4, 3), (5, 8), (6, 9), (7, 8), (8, 3), (9, 8)]
columns = ['Id', 'Random']

df3 = spark.createDataFrame(data,columns)
df3 = df3.withColumn("is_multiple_of_3", when(col("random") % 3 == 0, 1).otherwise(0))

df3.show()

+---+------+----------------+
| Id|Random|is_multiple_of_3|
+---+------+----------------+
|  0|     7|               0|
|  1|     6|               1|
|  2|     9|               1|
|  3|     7|               0|
|  4|     3|               1|
|  5|     8|               0|
|  6|     9|               1|
|  7|     8|               0|
|  8|     3|               1|
|  9|     8|               0|
+---+------+----------------+



In [21]:
from pyspark.sql import functions as F

data = [("john",), ("alice",), ("bob",)]
df = spark.createDataFrame(data,['name'])
df.show()
df_capitalized = df.withColumn('name',F.initcap("name"))

df_capitalized.show()

+-----+
| name|
+-----+
| john|
|alice|
|  bob|
+-----+

+-----+
| name|
+-----+
| John|
|Alice|
|  Bob|
+-----+



In [22]:
#9. How to calculate the number of characters in each word in a column
from pyspark.sql import functions as F

data = [("john",), ("alice",), ("bob",)]
df = spark.createDataFrame(data,['name'])

df_word_length = df.withColumn("word_length",F.length("name"))
df_word_length.show()

+-----+-----------+
| name|word_length|
+-----+-----------+
| john|          4|
|alice|          5|
|  bob|          3|
+-----+-----------+



In [27]:
from pyspark.sql.functions import col, count, when
data = [('A',1,None),('B',None,123),('B',3,456),('D',None,None)]
df = spark.createDataFrame(data,['Name','Value','Id'])
df.show()
null_counts =  df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show()

+----+-----+----+
|Name|Value|  Id|
+----+-----+----+
|   A|    1|NULL|
|   B| NULL| 123|
|   B|    3| 456|
|   D| NULL|NULL|
+----+-----+----+

+----+-----+---+
|Name|Value| Id|
+----+-----+---+
|   0|    2|  2|
+----+-----+---+

